<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/NEW_2026_10_02_%E2%80%94_Normalize_and_Join_Nested_JSON_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [8]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item (one item within one order), with that order's `order` number and `vendor_id` copied onto it.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [9]:
import pandas as pd

# record_path='lines' makes each line item its own row;
# meta copies the order-level fields onto every one of those rows
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
print(lines)

assert len(lines) == 4, 'expected 4 line items'
assert lines['qty'].sum() == 7, 'expected quantities to total 7'
print('Rows:', len(lines), '| Total qty:', lines['qty'].sum())

           item  qty order vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07
Rows: 4 | Total qty: 7


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [10]:
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True)
print(joined)

# rows that found no vendor are marked 'left_only' by indicator=True
unmatched = joined[joined['_merge'] == 'left_only']
print()
print('Unmatched vendor(s):', unmatched['vendor_id'].unique().tolist())
print('Units attached to unmatched vendor(s):', unmatched['qty'].sum())

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only

Unmatched vendor(s): ['V-07']
Units attached to unmatched vendor(s): 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [11]:
# I label the unmatched vendor as 'UNKNOWN (V-07)' instead of letting groupby drop it.
# If I left the name as NaN, groupby would silently skip those 3 units and the table
# would only add up to 4. Keeping the vendor_id in the label shows exactly which
# vendor is missing from the vendor list so it can be fixed at the source.
qty_by_vendor_df = joined.copy()
qty_by_vendor_df['name'] = qty_by_vendor_df['name'].fillna('UNKNOWN (' + qty_by_vendor_df['vendor_id'] + ')')

qty_by_vendor = qty_by_vendor_df.groupby('name')['qty'].sum().sort_values(ascending=False)
print(qty_by_vendor)
print('Total:', qty_by_vendor.sum())
assert qty_by_vendor.sum() == 7, 'every unit should still be accounted for'

name
Hoos Burgers      3
UNKNOWN (V-07)    3
Cav Merch         1
Name: qty, dtype: int64
Total: 7


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [12]:
# Default groupby drops rows where zone is NaN (the unmatched vendor V-07)
qty_by_zone = joined.groupby('zone')['qty'].sum()
print(qty_by_zone)
print('Top zone:', qty_by_zone.idxmax(), 'with', qty_by_zone.max(), 'items')
print('Zone total:', qty_by_zone.sum(), 'out of', joined['qty'].sum())

# Same table with dropna=False so the missing units show up as their own NaN row
print()
print(joined.groupby('zone', dropna=False)['qty'].sum())

# Zone A sold the most items (4). The zone totals add to 4, not 7.
# The missing 3 units are the Hot Dogs from order 3, vendor V-07, which is not in
# the vendor list, so it has no zone and groupby drops it by default.

zone
A    4
Name: qty, dtype: int64
Top zone: A with 4 items
Zone total: 4 out of 7

zone
A      4
NaN    3
Name: qty, dtype: int64


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [13]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# json_normalize crashes on order 5 because it has no 'lines' key.
# Fix: give any order without lines a single empty line ({}), so the order still
# shows up as a row instead of disappearing.
safe_json = [{**o, 'lines': o.get('lines') or [{}]} for o in ragged_json]

ragged = pd.json_normalize(safe_json, record_path='lines', meta=['order', 'vendor_id'])

# Nullable integer type: a missing qty shows as <NA>, which is NOT the same as 0
ragged['qty'] = ragged['qty'].astype('Int64')

# Label why each qty is present or missing
has_lines = {o['order']: 'lines' in o for o in ragged_json}
ragged['qty_status'] = 'present'
ragged.loc[ragged['qty'].isna(), 'qty_status'] = 'missing qty'
ragged.loc[~ragged['order'].map(has_lines), 'qty_status'] = 'no lines in order'

print(ragged)
print()
print('Known qty total:', ragged['qty'].sum(), '| rows with unknown qty:', ragged['qty'].isna().sum())

    item   qty order vendor_id         qty_status
0   Soda     2     4      V-01            present
1    NaN  <NA>     5      V-10  no lines in order
2  Fries  <NA>     6      V-01        missing qty

Known qty total: 2 | rows with unknown qty: 2


### Q6 — Validate

In [14]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

**a)** One row per line item means each row has just one item and one quantity, so it's easy to add up and group the numbers. For example, finding how many units each vendor or zone sold takes one `groupby`. With one row per order, the items are packed together in a list inside one cell, so you'd have to unpack them before you could do any math.

**b)** A quantity of 0 means we know nothing was sold. A missing quantity means we don't know how many were sold. If you treat missing as 0, the totals come out too low and you can't tell which data is missing. Keeping it as `<NA>` makes it clear which rows still need to be checked.